# CIDS · 19 — boundary refiner: small-case learning test (no val, no test)

Arms A and B left even their **training** cases unchanged (train-check 0.8000 → 0.801, where a perfect 8 mm band would reach 0.964). Before any further full run, this notebook checks whether the refiner can fix M0 on a **few real training cases**.

- **16 training cases** with both fixable missed tumour and fixable extra mask inside the 8 mm band.
- **64 fixed patches**, centred on the errors, with no augmentation. Each configuration trains from scratch for 1500 steps on the same patches.
- Configurations (frequency refiner):
  - loss `dice_bce` (current) vs `balanced` (Dice + BCE averaged per group: missed / extra / already correct);
  - output gate on vs off.

Every 250 steps it reports:
1. on the **same patches**: band Dice M0 → refined, the share of missed and extra voxels fixed, the share of correct voxels kept, and the residual pushed toward the truth (it must exceed 1.10 to flip a voxel);
2. on the **whole ROI** of the same 16 cases (sliding windows, as at inference);
3. on **16 unseen** training cases.

**Settings:** GPU T4 x2 · Internet ON. **Inputs:** cache + notebook 12 output. **Time:** ~30–40 min.
Run with **Save Version → Save & Run All**.


In [ ]:
REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = '/kaggle/input/datasets/kumarvemuri1/mama-mia-preprocessed-cache-2ch'
S3F_DIR    = '/kaggle/input'   # notebook 12 output is found automatically (folder with summary.json + masks/)
CODE_DIR   = '/kaggle/working/cids'
OUT        = '/kaggle/working/s7_learn'
CONFIGS    = 'freq:dice_bce:gate,freq:balanced:gate,freq:dice_bce:nogate,freq:balanced:nogate'
PATHS = [CACHE_ROOT]

In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

def sh(cmd):
    """Run a shell step, stream its output, and STOP the notebook if it fails (plain !cmd does not)."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}): {cmd}')

In [ ]:
sh(f'pip install -q monai medim einops appdirs')
sh(f'python -m pytest -q tests')

In [ ]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
for q in PATHS:
    print('OK ' if Path(q).exists() else '!! MISSING', q)
assert all(Path(q).exists() for q in PATHS)
hits = [p.parent for p in Path(S3F_DIR).rglob('summary.json') if (p.parent / 'masks').is_dir() or (p.parent / 'masks.zip').exists()]
print('notebook 12 output:', hits)
assert hits, 'attach the notebook 12 output (Add Input -> Your Work -> Notebooks -> 12_s3f_jury_masks)'
S3F_DIR = str(hits[0])

## Smoke test (~3 min): 2 configurations, 40 steps


In [ ]:
sh(f'python scripts/s7_learn_test.py --s3f_dir {S3F_DIR} --cache_root {CACHE_ROOT} --work /kaggle/working/learn_smoke/rois --out /kaggle/working/learn_smoke/out --pool 80 --n_cases 4 --n_unseen 2 --steps 40 --eval_every 20 --configs freq:dice_bce:gate,freq:balanced:nogate')
sh('rm -rf /kaggle/working/learn_smoke')


## Full learning test
**Read the SUMMARY table at the end:**

| Result | Meaning |
|---|---|
| `patch_gain` ≈ 0 | It cannot fit even fixed examples: optimisation / loss problem |
| `patch_gain` > 0 but `roi_gain` ≈ 0 | Inference problem (patch context, sliding-window blending) |
| `roi_gain` > 0 but `unseen_gain` ≤ 0 | It memorises; corrections don't transfer from the image alone |
| all three > 0 | Ready for a full run with this configuration |


In [ ]:
sh(f'python scripts/s7_learn_test.py --s3f_dir {S3F_DIR} --cache_root {CACHE_ROOT} --out {OUT} --configs {CONFIGS}')


In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open(f'{OUT}/learn_summary.json'))['summary'], indent=2))
print(pd.read_csv(f'{OUT}/learn_log.csv').to_string())
